In [ ]:
#| default_exp magic/network

In [ ]:
#| export
from __future__ import annotations
from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
import io, base64, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import NamedTuple
from enum import Enum
from collections import defaultdict
import random
import math
import uuid
import httpx

import networkx as nx

from monsterui.all import *
from fastcore.basics import patch


In [ ]:
#| export
from fasthtml.common import *
from fasthtml.jupyter import *
from fasthtml.common import *

In [ ]:
#| export
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.primitives import MapRect, MapPath,MapCord,  MapSize, HexRegion
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.game.flag import OverlaySpec

In [ ]:
#| export
from HexMagic.magic.board import GameParts
from HexMagic.magic.food import FOOD_TABLE, FoodStats
from HexMagic.magic.pieces import Piece, Squad
from HexMagic.game.flag import CountryFlag , PieceType, GameContext, DiagramGlyphs

!cat ../../HexMagic/magic/pieces.py

In [ ]:
#| export
@patch
def to_nx(self: Squad) -> nx.DiGraph:
    """Build directed sight graph as NetworkX DiGraph with food attributes."""
    G = nx.DiGraph()
    placed = [p for p in self.pieces if p.place is not None]
    parts = placed[0].place.parent if placed else None
    if not parts:
        return G
    
    for p in placed:
        idx = parts.hp2i(p.place.location)
        G.add_node(p.id, 
                   piece=p,  # back-reference
                   piece_type=p.type.value,
                   harvest=int(p.food.harvest * p.place.harvest),
                   diet=p.food.diet,
                   drain=p.food.drain,
                   cost=p.food.cost,
                   pantry=p.place.pantry,
                   hex_idx=idx,
                   is_sink=p.food.harvest == 0.0 or p.type == PieceType.KING,
                   is_king=p.type == PieceType.KING)
    
    for p in placed:
        cone = p.sight().hexes
        for other in placed:
            if other is p:
                continue
            idx = parts.hp2i(other.place.location)
            if idx in cone:
                G.add_edge(p.id, other.id)
    
    return G

In [ ]:
#| export
@patch
def supply_bottlenecks(self: Squad) -> list[Piece]:
    """Pieces whose removal disconnects the sight network."""
    G = self.to_nx()
    U = G.to_undirected(as_view=True)
    return [G.nodes[n]['piece'] for n in nx.articulation_points(U)]

@patch
def isolated_pieces(self: Squad) -> list[Piece]:
    """Pieces not connected to the king's component."""
    G = self.to_nx()
    U = G.to_undirected(as_view=True)
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id: return []
    king_component = nx.node_connected_component(U, king_id)
    return [G.nodes[n]['piece'] for n in G.nodes if n not in king_component]

In [ ]:
#| export
@patch
def food_sinks(self: Squad) -> list[Piece]:
    """Pieces that consume but don't produce — need to be fed by the network."""
    G = self.to_nx()
    return [G.nodes[n]['piece'] for n in G.nodes 
            if G.nodes[n]['is_sink']]

@patch
def food_sources(self: Squad) -> list[Piece]:
    """Pieces that can produce surplus (harvest > diet)."""
    G = self.to_nx()
    return [G.nodes[n]['piece'] for n in G.nodes 
            if G.nodes[n]['harvest'] > G.nodes[n]['diet']]


In [ ]:
#| export
@patch
def flow_network(self: Squad) -> dict:
    """Analyse food flow using the sight graph."""
    G = self.to_nx()
    if not G.nodes:
        return {'G': G, 'sinks': set(), 'reachable': {},
                'disconnected': [], 'sink_layers': {}}

    sinks = {n for n, d in G.nodes(data=True) if d['is_sink']}

    R = G.reverse(copy=False)
    sink_connected = {}
    for sink in sinks:
        sink_connected[sink] = nx.ancestors(R, sink) | {sink}

    reachable = {n: set() for n in G.nodes}
    for sink, connected in sink_connected.items():
        for n in connected:
            reachable[n].add(sink)

    disconnected = [G.nodes[n]['piece'] for n in G.nodes
                    if not reachable[n] and n not in sinks]

    sink_layers = {}
    for sink in sinks:
        sink_layers[sink] = nx.shortest_path_length(R, sink)

    return {
        'G': G,
        'sinks': sinks,
        'sink_connected': sink_connected,
        'sink_layers': sink_layers,
        'reachable': reachable,
        'disconnected': disconnected,
    }

In [ ]:
#| export
@patch
def resolve_flow(self: Squad):
    """Centralized treasury with local fallback for disconnected pieces."""
    G = self.to_nx()
    if len(G) < 2:
        return

    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id:
        return

    king_piece = G.nodes[king_id]['piece']
    U = G.to_undirected(as_view=True)
    king_comp = nx.node_connected_component(U, king_id)

    for nid, data in G.nodes(data=True):
        if nid == king_id:
            continue
        p = data['piece']
        if nid in king_comp:
            # Connected: everything flows to king
            king_piece.place.pantry += p.place.pantry
            p.place.pantry = 0
        # else: disconnected, keeps its own pantry (will likely starve)

## Squad Flow

In [ ]:
#| export
@dataclass
class SquadFlow:
    """Computed food flow state for a squad. Shared by diagram + overlay renderers."""
    squad: Squad
    G: nx.DiGraph
    edge_flows: dict  # (u, v) -> float
    node_inflow: dict  # nid -> float
    node_outflow: dict  # nid -> float

    @classmethod
    def topology_only(cls, squad: Squad) -> 'SquadFlow':
        """Build graph topology without running max-flow. Fast."""
        G = squad.to_nx()
        empty = {n: 0 for n in G.nodes}
        return cls(squad, G, {}, empty, empty)
    
    @classmethod
    def from_squad(cls, squad: Squad) -> 'SquadFlow':
        G = squad.to_nx()
        if len(G) < 2:
            empty = {n: 0 for n in G.nodes}
            return cls(squad, G, {}, empty, empty)

        king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
        if not king_id:
            empty = {n: 0 for n in G.nodes}
            return cls(squad, G, {}, empty, empty)

        U = G.to_undirected(as_view=True)
        king_comp = nx.node_connected_component(U, king_id)

        edge_flows = {}
        node_inflow = {n: 0 for n in G.nodes}
        node_outflow = {n: 0 for n in G.nodes}

        for nid, data in G.nodes(data=True):
            if nid == king_id or nid not in king_comp:
                continue
            net = data['harvest'] - data['diet']
            if net > 0:
                # Surplus flows from this piece toward king
                # Find the shortest sight-path to king for the diagram
                try:
                    path = nx.shortest_path(U, nid, king_id)
                    for i in range(len(path) - 1):
                        u, v = path[i], path[i + 1]
                        edge_flows[(u, v)] = edge_flows.get((u, v), 0) + net
                        node_outflow[u] += net
                        node_inflow[v] += net
                except nx.NetworkXNoPath:
                    pass
            elif net < 0:
                # Deficit: king sends food to this piece
                try:
                    path = nx.shortest_path(U, king_id, nid)
                    for i in range(len(path) - 1):
                        u, v = path[i], path[i + 1]
                        edge_flows[(u, v)] = edge_flows.get((u, v), 0) + abs(net)
                        node_outflow[u] += abs(net)
                        node_inflow[v] += abs(net)
                except nx.NetworkXNoPath:
                    pass

        return cls(squad, G, edge_flows, node_inflow, node_outflow)
    
    def post_net(self, nid) -> float:
        """Net food for a node after flow redistribution."""
        data = self.G.nodes[nid]
        return data['harvest'] - data['diet'] + self.node_inflow[nid] - self.node_outflow[nid]
    
    def edge_flow(self, u, v) -> float:
        """Net flow on undirected edge (positive = u→v)."""
        return self.edge_flows.get((u, v), 0) - self.edge_flows.get((v, u), 0)
    
    @property
    def max_flow(self) -> float:
        return max(self.edge_flows.values(), default=1)
    
    @property
    def total_flow(self) -> float:
        return sum(self.edge_flows.values())
    
    @property
    def bottlenecks(self) -> set:
        if len(self.G) < 2: return set()
        return set(nx.articulation_points(self.G.to_undirected(as_view=True)))
    
    @property 
    def total_surplus(self) -> int:
        placed = [p for p in self.squad.pieces if p.place is not None]
        return sum(int(p.food.harvest * p.place.harvest) for p in placed) - sum(p.food.diet for p in placed)

    # Inside SquadFlow
    @property
    def king_component(self) -> set:
        if len(self.G) < 2: return set(self.G.nodes)
        king_id = next((n for n, d in self.G.nodes(data=True) if d['is_king']), None)
        if not king_id: return set()
        U = self.G.to_undirected(as_view=True)
        return nx.node_connected_component(U, king_id)


In [ ]:
#| export
@patch
def sight_projection(self: SquadFlow, parts: GameParts) -> float:
    """How well do sight cones project toward the squad's target hex?"""
    target = getattr(self.squad, '_target', None)
    if target is None or target.hex_idx < 0:
        return 0.0
    target_hp = parts.i2hp(target.hex_idx)
    score = 0.0
    for n in self.G.nodes:
        cone = self.G.nodes[n]['piece'].sight().hexes
        if target.hex_idx in cone:
            score += 10.0
        elif cone:
            min_dist = min(parts.i2hp(h).distance(target_hp) for h in cone)
            score += 2.0 / max(1, min_dist)
    return score


In [ ]:
#| export
@dataclass
class SquadFlowRenderer:
    """Pure SVG fragment generator for squad food flow.
    Shared by SquadFlowDiagram (panel) and FoodNetworkOverlay (map layer).
    Position source is injected — caller decides where nodes live."""
    
    flag: 'CountryFlag'
    node_r: float = 20.0
    edge_opacity: float = 0.6

    # ── Edge helpers ────────────────────────────────────────────────

    def edge_color_width(self, flow: SquadFlow, u, v) -> tuple[str, float, str]:
        """Returns (color, stroke_width, dash) for an undirected edge."""
        net = abs(flow.edge_flow(u, v))
        is_bridge = u in flow.bottlenecks or v in flow.bottlenecks

        if net > 0:
            return "#4CAF50", 1.5 + (net / flow.max_flow) * 4, ""
        elif is_bridge:
            return "#E53935", 1.5, "4,3"
        else:
            return self.flag.darkPrimary, 1.0, "4,3"

    def edge_fragment(self, flow: SquadFlow,
                      u, v,
                      p1: MapCord, p2: MapCord,
                      builder,
                      shrink: float = 30.0) -> str:
        """SVG for one undirected edge: arrow + optional flow label."""
        dist = p1.distance(p2)
        if dist < 1: return ""

        net = flow.edge_flow(u, v)
        total = abs(net)
        src_p, dst_p = (p1, p2) if net >= 0 else (p2, p1)

        dx, dy = (dst_p.x - src_p.x) / dist, (dst_p.y - src_p.y) / dist
        start = MapCord(src_p.x + dx * shrink, src_p.y + dy * shrink)
        end   = MapCord(dst_p.x - dx * shrink, dst_p.y - dy * shrink)

        color, sw, dash = self.edge_color_width(flow, u, v)
        opacity = self.edge_opacity if total > 0 else self.edge_opacity * 0.4

        style = StyleCSS(f"sfr_edge_{u[:4]}_{v[:4]}",
                         stroke=color, stroke_width=sw,
                         fill=color, opacity=opacity,
                         **{"stroke-dasharray": dash} if dash else {})
        builder.add_style(style)

        parts = [MapPath([start, end], style).with_arrowhead(arrow_size=10, arrow_angle=25)]

        if total > 0:
            mid = start.midPoint(end)
            px, py = -dy * 12, dx * 12
            parts.append(
                f'<text x="{mid.x + px:.1f}" y="{mid.y + py:.1f}" '
                f'text-anchor="middle" font-size="10" font-weight="bold" '
                f'fill="#4CAF50" opacity="0.8">{total:.0f}🌾</text>')

        return '\n'.join(str(p) for p in parts)

    def all_edges(self, flow: SquadFlow,
                  svg_pos: dict[str, MapCord],
                  builder,
                  shrink: float = 30.0) -> str:
        """All edges, deduped."""
        seen = set()
        parts = []
        for u, v in flow.G.edges():
            key = tuple(sorted([u, v]))
            if key in seen: continue
            seen.add(key)
            parts.append(self.edge_fragment(flow, u, v,
                                            svg_pos[u], svg_pos[v],
                                            builder, shrink))
        return '\n'.join(parts)

    # ── Node helpers ─────────────────────────────────────────────────

    def node_ring_fragment(self, nid, data, flow: SquadFlow,
                           center: MapCord, builder) -> str:
        """Bottleneck outer ring + colored inner ring."""
        glyphs = DiagramGlyphs(self.flag, size=self.node_r)
        glyphs.register_styles(builder)
        ring_r = self.node_r + 6
        parts = []

        if nid in flow.bottlenecks:
            danger = glyphs.danger_style()
            builder.add_style(danger)
            parts.append(glyphs.ring(center, ring_r + 4, danger))

        post_net = flow.post_net(nid)
        ring_color = ("#FFD700" if data['is_king']
                      else "#4CAF50" if post_net >= 0
                      else "#E53935")
        ring_style = StyleCSS(f"sfr_ring_{nid[:6]}",
                              fill="white", stroke=ring_color,
                              stroke_width=3.5 if data['is_king'] else 2.5)
        builder.add_style(ring_style)
        parts.append(glyphs.ring(center, ring_r, ring_style))
        return '\n'.join(parts)

    def node_food_pip(self, nid, flow: SquadFlow,
                      center: MapCord, builder) -> str:
        """Food pip in upper-right of node."""
        glyphs = DiagramGlyphs(self.flag, size=self.node_r)
        glyphs.register_styles(builder)
        ring_r = self.node_r + 6
        post_net = flow.post_net(nid)
        frac = min(1.0, max(0.0, (post_net + 10) / 20))
        pip_center = MapCord(center.x + ring_r * 0.7, center.y - ring_r * 0.7)
        return glyphs.food_pip(pip_center, frac, max_r=8, min_r=4)

    def node_labels(self, piece, flow: SquadFlow, nid,
                    center: MapCord) -> str:
        """Name + stats text below node."""
        ring_r = self.node_r + 6
        post_net = flow.post_net(nid)
        pantry = piece.place.pantry if piece.place else 0
        tenure = piece.place.tenure if piece.place else 0
        life = piece.food.lifespan
        turns_left = max(0, life - tenure) if life < 999 else -1
        turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
        name_y = center.y + ring_r + 14
        net_color = "#4CAF50" if post_net >= 0 else "#E53935"

        parts = [
            f'<text x="{center.x:.1f}" y="{name_y:.1f}" '
            f'text-anchor="middle" font-size="14" fill="{self.flag.darkPrimary}">'
            f'{piece.name}</text>',
            f'<text x="{center.x:.1f}" y="{name_y+16:.1f}" '
            f'text-anchor="middle" font-size="12">'
            f'<tspan fill="{net_color}" font-size="15" font-weight="700">{post_net:+.0f}🌾</tspan> '
            f'<tspan fill="{self.flag.darkPrimary}" font-size="15" font-weight="700">🧺{pantry}</tspan> '
            f'<tspan fill="#666">{turns_str}</tspan></text>'
        ]

        return '\n'.join(parts)


In [ ]:
#| export
@patch
def node_labels(self:SquadFlowRenderer, piece, flow: SquadFlow, nid, center: MapCord) -> str:
    """Name + vertically stacked SVG pill badges below node."""
    ring_r   = self.node_r + 6
    post_net = flow.post_net(nid)
    pantry   = piece.place.pantry if piece.place else 0
    tenure   = piece.place.tenure if piece.place else 0
    life     = piece.food.lifespan
    turns_left = max(0, life - tenure) if life < 999 else -1
    turns_str  = "∞" if turns_left < 0 else f"{turns_left}t"

    net_color  = "#4CAF50" if post_net >= 0 else "#E53935"
    net_bg     = "#e8f5e9" if post_net >= 0 else "#ffebee"
    pant_color = self.flag.darkPrimary

    cx = center.x
    y  = center.y + ring_r + 18  # start below ring
    dy = 22                       # vertical spacing between badges

    def pill(label, fg, bg, y_pos, w=70, h=16, r=8):
        return (
            f'<rect x="{cx - w/2:.1f}" y="{y_pos - h/2 - 1:.1f}" '
            f'width="{w}" height="{h}" rx="{r}" fill="{bg}" opacity="0.85"/>'
            f'<text x="{cx:.1f}" y="{y_pos:.1f}" text-anchor="middle" '
            f'dominant-baseline="central" font-size="12" font-weight="700" fill="{fg}">'
            f'{label}</text>'
        )

    parts = [
        # Name — no pill, just bold text
        f'<text x="{cx:.1f}" y="{y:.1f}" text-anchor="middle" '
        f'dominant-baseline="central" font-size="13" font-weight="700" '
        f'fill="{self.flag.darkPrimary}">{piece.name}</text>',

        # Net food pill
        pill(f"{post_net:+.0f} 🍖", net_color, net_bg, y + dy),

        # Pantry pill
        pill(f"🧺 {pantry}", pant_color, "#f5f5f5", y + dy * 2),

        # Turns pill — only show if finite
        *([] if turns_left < 0 else
          [pill(f"⏳ {turns_str}", "#666", "#f5f5f5", y + dy * 3, w=55)]),
    ]

    return '\n'.join(parts)


In [ ]:
??CountryFlag.pantry_badge 


In [ ]:
??CountryFlag.cost_badge

can we update to use badges

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

There's a small mismatch to flag first: `node_labels` currently returns **SVG strings** (it lives inside an SVG diagram), while `pantry_badge` and `cost_badge` return **`FT` HTML components**. They can't be directly mixed.

You have two paths:

1. **SVG badge helpers on `CountryFlag`** — add `pantry_badge_svg` / `cost_badge_svg` that return `<text>` or `<rect>+<text>` SVG fragments, then use them in `node_labels`. Keeps everything in the SVG world.

2. **Move `node_labels` to HTML** — if the diagram panel is HTML-based (not pure SVG), render node labels as `FT` using the existing badges, possibly using `foreignObject` to embed inside SVG.

Which fits better? A couple of questions:

- Is `SquadFlowDiagram` rendered as a standalone SVG, or is it inside an HTML panel where mixing SVG + HTML is easy?
- Do `pantry_badge` / `cost_badge` already appear elsewhere in the UI so you want visual consistency with those exact components?

If it's mostly SVG, option 1 is cleanest — I can write `pantry_badge_svg` and `cost_badge_svg` patch methods on `CountryFlag` that mirror the style of the HTML versions (emoji + amount, color-coded by affordability), then update `node_labels` to call them.

So this  f'<tspan fill="#888">📦{pantry}</tspan> ' should have the correct emoji and large font similar to the component

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at `pantry_badge`, it uses 🧺 (basket emoji) not 📦, and `font-size:1.2rem` with `font-weight:700`. In SVG terms that's roughly `font-size="19"` and bold. Here's the updated tspan:

```python
f'<tspan fill="{self.flag.darkPrimary}" font-size="16" font-weight="700">🧺{pantry}</tspan> '
```

And the full updated `node_labels` lines for context:

```python
parts = [
    f'<text x="{center.x:.1f}" y="{name_y:.1f}" '
    f'text-anchor="middle" font-size="14" fill="{self.flag.darkPrimary}">'
    f'{piece.name}</text>',
    f'<text x="{center.x:.1f}" y="{name_y+16:.1f}" '
    f'text-anchor="middle" font-size="12">'
    f'<tspan fill="{net_color}" font-size="15" font-weight="700">{post_net:+.0f}🌾</tspan> '
    f'<tspan fill="{self.flag.darkPrimary}" font-size="15" font-weight="700">🧺{pantry}</tspan> '
    f'<tspan fill="#666">{turns_str}</tspan></text>'
]
```

Key changes:
- 📦 → 🧺 (matches `pantry_badge`)
- `font-size="15"` + `font-weight="700"` on both the food net and pantry tspans
- Pantry uses `self.flag.darkPrimary` for color (matches the badge)
- Bumped the `y` offset slightly (`+16` instead of `+13`) to give the larger text room

### Testing

In [ ]:
myStuff = GameParts()

In [ ]:
myStuff.loadSquads()

In [ ]:


ourTroop = myStuff.squads[0]
#show(ourTroop.pieces[0])



In [ ]:
ourTroop.table()

How do we build a good test for our network code. I feel we should have some sort of table,

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great instinct! A table-driven approach works really well for network/graph code because you can define several small "scenarios" (different piece placements) and assert expected outcomes for each.

Before diving in, a couple of questions to narrow the focus:

1. **What kind of table** are you thinking — a Python list of dicts/`NamedTuple`s, a `pytest.mark.parametrize` table, or something displayed visually (like a pandas DataFrame)?
2. **Which functions** do you most want to cover first — e.g. `supply_bottlenecks`, `isolated_pieces`, `SquadFlow.from_squad`, or all of them?

The tricky part with your code is that the graph topology depends on piece *placement* (hex positions + sight cones), which requires a live `GameParts` board. So a key design question is: **do you want to mock the sight graph directly** (fast, isolated unit tests), or **place pieces on a real board** (more integration-style)?

Lets figure out how to mock sight so we can have connections.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at `to_nx()`, edges are built from `p.sight().hexes` — a set of hex indices. So the minimal mock is: **patch `sight()` on each piece to return a `HexRegion` with a fixed set of indices**, without needing any board at all.

The key line in `to_nx` is:
```python
cone = p.sight().hexes
```

So if you can make `p.sight()` return something with a `.hexes` attribute (just a plain set of ints), you bypass all the board/placement machinery.

Two questions:
1. Does `HexRegion` accept an arbitrary set of hex indices, or would a simple mock object (with just a `.hexes` attribute) be easier?
2. Do you want to patch using `unittest.mock.patch`, or just directly assign a lambda/property on the piece object before testing?

In [ ]:
!cat ../../HexMagic/plot/*.py

This could give you some info about HexRegions. They are just sets

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's the key insight! Since `HexRegion.hexes` is just a `set[int]`, and `to_nx()` only accesses `p.sight().hexes`, the mock can be dead simple:

```python
class FakeSight:
    def __init__(self, indices): self.hexes = set(indices)
```

Then you can write a scenario like:

```python
# Piece 0 can see piece 1, piece 1 can see piece 2, etc.
piece0.sight = lambda: FakeSight([idx1])
piece1.sight = lambda: FakeSight([idx0, idx2])
```

Does that feel right to you? And do you still want to mock `p.place` too (since `to_nx` also accesses `p.place.parent.hp2i(p.place.location)`)?

This seems great. we could figure out which members we want to link and then build FakeSight for it to construct

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly! So you'd define the topology as something like:

```python
# adjacency: piece_index -> [list of piece indices it can see]
connections = {0: [1], 1: [0, 2], 2: [1]}
```

Then build `FakeSight` for each piece from that. 

Do you also need to mock `p.place` for `to_nx` to work, or do you want to refactor `to_nx` to accept the graph topology directly?

Can you build a test of our graph that uses FakeSight

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at `to_nx()`, it also reads `p.place.parent.hp2i(...)`, `p.place.harvest`, `p.place.pantry`, and `p.place.level`, so we need a thin `FakePlace` + `FakeParts` too. Here's a self-contained test:

```python
from dataclasses import dataclass

class FakeSight:
    def __init__(self, indices): self.hexes = set(indices)

class FakeParts:
    """hp2i is identity — location IS the hex index."""
    def hp2i(self, loc): return loc
    def i2hp(self, idx): return idx

class FakePlace:
    def __init__(self, idx, parent):
        self.location = idx
        self.parent = parent
        self.level = 1
        self.harvest = 1.0
        self.pantry = 0
        self.facing = HexPosition.E
        self.tenure = 0

def wire_squad(types, connections):
    """Build a Squad with fake placement and sight from an adjacency dict.
    
    Args:
        types: list of PieceType, one per piece
        connections: dict[int, list[int]] — piece index -> visible piece indices
    
    Returns:
        Squad with pieces[i] at hex index i, sight wired per connections
    """
    flag = CountryFlag.seaborn("husl", 1)[0]
    parts = FakeParts()
    
    pieces = []
    for i, pt in enumerate(types):
        p = Piece(id=f"p{i}", flag=flag, type=pt, name=f"Test{i}", year=1900+i)
        p.place = FakePlace(idx=i, parent=parts)
        pieces.append(p)
    
    # Wire sight: piece i sees hex indices of connected pieces
    for i, p in enumerate(pieces):
        visible_hexes = connections.get(i, [])
        p.sight = lambda hexes=visible_hexes: FakeSight(hexes)
    
    squad = Squad(pieces=pieces, name="TestSquad", flag=flag)
    return squad

# === Tests ===

def test_linear_chain():
    """0 -> 1 -> 2 (king). Piece 0 can't see king directly."""
    sq = wire_squad(
        [PieceType.PAWN, PieceType.PAWN, PieceType.KING],
        {0: [1], 1: [0, 2], 2: [1]}
    )
    G = sq.to_nx()
    assert len(G.nodes) == 3
    assert len(G.edges) == 4  # bidirectional: 0↔1, 1↔2
    assert G.has_edge("p0", "p1")
    assert not G.has_edge("p0", "p2")  # no direct sight
    assert sq.supply_bottlenecks()[0].id == "p1"  # p1 is the bridge
    assert sq.isolated_pieces() == []
    print("✅ linear chain")

def test_disconnected_piece():
    """Piece 2 sees nobody — isolated from king."""
    sq = wire_squad(
        [PieceType.KING, PieceType.PAWN, PieceType.PAWN],
        {0: [1], 1: [0], 2: []}
    )
    iso = sq.isolated_pieces()
    assert len(iso) == 1 and iso[0].id == "p2"
    print("✅ disconnected piece")

def test_fully_connected():
    """Everyone sees everyone — no bottlenecks."""
    sq = wire_squad(
        [PieceType.KING, PieceType.PAWN, PieceType.PAWN],
        {0: [1, 2], 1: [0, 2], 2: [0, 1]}
    )
    assert sq.supply_bottlenecks() == []
    assert sq.isolated_pieces() == []
    print("✅ fully connected")

def test_food_sinks():
    """King is always a sink."""
    sq = wire_squad(
        [PieceType.KING, PieceType.PAWN],
        {0: [1], 1: [0]}
    )
    sinks = sq.food_sinks()
    assert any(s.type == PieceType.KING for s in sinks)
    print("✅ food sinks")

test_linear_chain()
test_disconnected_piece()
test_fully_connected()
test_food_sinks()
```

The `wire_squad` helper is the key — you just pass piece types and a connections dict, and it builds everything `to_nx()` needs. Try running it and see if the assertions hold!

In [ ]:
from dataclasses import dataclass

class FakeSight:
    def __init__(self, indices): self.hexes = set(indices)

class FakeParts:
    """hp2i is identity — location IS the hex index."""
    def hp2i(self, loc): return loc
    def i2hp(self, idx): return idx

class FakePlace:
    def __init__(self, idx, parent):
        self.location = idx
        self.parent = parent
        self.level = 1
        self.harvest = 1.0
        self.pantry = 0
        self.facing = HexPosition.E
        self.tenure = 0

def wire_squad(types, connections):
    """Build a Squad with fake placement and sight from an adjacency dict.
    
    Args:
        types: list of PieceType, one per piece
        connections: dict[int, list[int]] — piece index -> visible piece indices
    
    Returns:
        Squad with pieces[i] at hex index i, sight wired per connections
    """
    flag = CountryFlag.seaborn("husl", 1)[0]
    parts = FakeParts()
    
    pieces = []
    for i, pt in enumerate(types):
        p = Piece(id=f"p{i}", flag=flag, type=pt, name=f"Test{i}", year=1900+i)
        p.place = FakePlace(idx=i, parent=parts)
        pieces.append(p)
    
    # Wire sight: piece i sees hex indices of connected pieces
    for i, p in enumerate(pieces):
        visible_hexes = connections.get(i, [])
        p.sight = lambda hexes=visible_hexes: FakeSight(hexes)
    
    squad = Squad(pieces=pieces, name="TestSquad", flag=flag)
    return squad

# === Tests ===

def test_linear_chain():
    """0 -> 1 -> 2 (king). Piece 0 can't see king directly."""
    sq = wire_squad(
        [PieceType.PAWN, PieceType.PAWN, PieceType.KING],
        {0: [1], 1: [0, 2], 2: [1]}
    )
    G = sq.to_nx()
    assert len(G.nodes) == 3
    assert len(G.edges) == 4  # bidirectional: 0↔1, 1↔2
    assert G.has_edge("p0", "p1")
    assert not G.has_edge("p0", "p2")  # no direct sight
    assert sq.supply_bottlenecks()[0].id == "p1"  # p1 is the bridge
    assert sq.isolated_pieces() == []
    print("✅ linear chain")

def test_disconnected_piece():
    """Piece 2 sees nobody — isolated from king."""
    sq = wire_squad(
        [PieceType.KING, PieceType.PAWN, PieceType.PAWN],
        {0: [1], 1: [0], 2: []}
    )
    iso = sq.isolated_pieces()
    assert len(iso) == 1 and iso[0].id == "p2"
    print("✅ disconnected piece")

def test_fully_connected():
    """Everyone sees everyone — no bottlenecks."""
    sq = wire_squad(
        [PieceType.KING, PieceType.PAWN, PieceType.PAWN],
        {0: [1, 2], 1: [0, 2], 2: [0, 1]}
    )
    assert sq.supply_bottlenecks() == []
    assert sq.isolated_pieces() == []
    print("✅ fully connected")

def test_food_sinks():
    """King is always a sink."""
    sq = wire_squad(
        [PieceType.KING, PieceType.PAWN],
        {0: [1], 1: [0]}
    )
    sinks = sq.food_sinks()
    assert any(s.type == PieceType.KING for s in sinks)
    print("✅ food sinks")

test_linear_chain()
test_disconnected_piece()
test_fully_connected()
test_food_sinks()


In [ ]:
@patch
def pantry_bar(piece, flow, nid, center: MapCord, radius: float = 25) -> str:
    """Pantry fill bar — map overlay only (diagram uses footer instead)."""
    pantry   = piece.place.pantry if piece.place else 0
    post_net = flow.post_net(nid)
    bar_w    = radius * 0.8
    bar_h    = max(3, radius * 0.1)
    bar_x    = center.x - bar_w / 2
    bar_y    = center.y + radius * 0.5 * 2.0
    max_p    = max(piece.food.cost * 2, 20)
    frac     = max(0.0, min(1.0, pantry / max_p))
    color    = "#4CAF50" if pantry > 0 else "#E53935"

    return (f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" width="{bar_w:.1f}" '
            f'height="{bar_h:.1f}" rx="1.5" fill="#ddd" opacity="0.6"/>'
            f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" width="{bar_w*frac:.1f}" '
            f'height="{bar_h:.1f}" rx="1.5" fill="{color}" opacity="0.8"/>')

In [ ]:
??Squad.table

Can we do something like squad table but that it includes the pantry bar and a pieces sources and sinks?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at `Squad.table` and `SquadFlow`, here's a `flow_table` that adds per-piece food stats, source/sink role, and pantry as a simple HTML progress bar:

```python
@patch
def flow_table(self: Squad):
    flow = SquadFlow.from_squad(self)
    G = flow.G
    king_comp = flow.king_component
    sinks = flow.sinks

    rows = []
    for p in self.pieces:
        if p.place is None:
            continue
        nid = p.id
        data = G.nodes[nid]
        
        harvest  = data['harvest']
        diet     = data['diet']
        net      = harvest - diet
        pantry   = p.place.pantry
        post     = flow.post_net(nid)
        
        # Role badge
        if data['is_king']:   role = "👑 King"
        elif nid in sinks:    role = "🍽️ Sink"
        elif net > 0:         role = "🌾 Source"
        else:                 role = "⚖️ Neutral"
        
        # Connected to king?
        connected = "✅" if nid in king_comp else "❌ Isolated"
        
        # Pantry bar as inline HTML
        max_p  = max(p.food.cost * 2, 20)
        frac   = max(0.0, min(1.0, pantry / max_p))
        color  = "#4CAF50" if pantry > 0 else "#E53935"
        bar = (f'<div style="background:#ddd;border-radius:3px;width:60px;height:8px">'
               f'<div style="background:{color};width:{frac*60:.0f}px;height:8px;border-radius:3px"></div>'
               f'</div><small>{pantry}</small>')

        rows.append(Tr(
            Td(p.icon()),
            Td(p.name, cls=TextT.medium),
            Td(role),
            Td(f"🌾{harvest} 🍽️{diet} {'▲' if net>=0 else '▼'}{abs(net)}"),
            Td(NotStr(bar)),
            Td(f"{post:+.0f}", style=f"color:{'green' if post>=0 else 'red'}"),
            Td(connected),
        ))

    return Card(
        Table(
            Thead(Tr(
                Th(""), Th("Name"), Th("Role"),
                Th("Food"), Th("Pantry"), Th("Net"), Th("Link")
            )),
            Tbody(*rows),
        ),
        header=(
            H4(self.name),
            P(f"Total surplus: {flow.total_surplus:+d} · Flow: {flow.total_flow:.0f}", 
              cls=TextPresets.muted_sm)
        ),
    )
```

Try calling `ourTroop.flow_table()` — the columns are:
- **Role** — King / Source / Sink / Neutral
- **Food** — harvest, diet, net delta
- **Pantry** — visual bar + raw value
- **Net** — post-flow balance (green/red)
- **Link** — connected to king or isolated

In [ ]:
@patch
def flow_table(self: Squad):
    flow = SquadFlow.from_squad(self)
    G = flow.G
    king_comp = flow.king_component
    sinks = flow.sinks

    rows = []
    for p in self.pieces:
        if p.place is None:
            continue
        nid = p.id
        data = G.nodes[nid]
        
        harvest  = data['harvest']
        diet     = data['diet']
        net      = harvest - diet
        pantry   = p.place.pantry
        post     = flow.post_net(nid)
        
        # Role badge
        if data['is_king']:   role = "👑 King"
        elif nid in sinks:    role = "🍽️ Sink"
        elif net > 0:         role = "🌾 Source"
        else:                 role = "⚖️ Neutral"
        
        # Connected to king?
        connected = "✅" if nid in king_comp else "❌ Isolated"
        
        # Pantry bar as inline HTML
        max_p  = max(p.food.cost * 2, 20)
        frac   = max(0.0, min(1.0, pantry / max_p))
        color  = "#4CAF50" if pantry > 0 else "#E53935"
        bar = (f'<div style="background:#ddd;border-radius:3px;width:60px;height:8px">'
               f'<div style="background:{color};width:{frac*60:.0f}px;height:8px;border-radius:3px"></div>'
               f'</div><small>{pantry}</small>')

        rows.append(Tr(
            Td(p.icon()),
            Td(p.name, cls=TextT.medium),
            Td(role),
            Td(f"🌾{harvest} 🍽️{diet} {'▲' if net>=0 else '▼'}{abs(net)}"),
            Td(NotStr(bar)),
            Td(f"{post:+.0f}", style=f"color:{'green' if post>=0 else 'red'}"),
            Td(connected),
        ))

    return Card(
        Table(
            Thead(Tr(
                Th(""), Th("Name"), Th("Role"),
                Th("Food"), Th("Pantry"), Th("Net"), Th("Link")
            )),
            Tbody(*rows),
        ),
        header=(
            H4(self.name),
            P(f"Total surplus: {flow.total_surplus:+d} · Flow: {flow.total_flow:.0f}", 
              cls=TextPresets.muted_sm)
        ),
    )


In [ ]:
@patch
def flow_table(self: Squad):
    flow = SquadFlow.from_squad(self)
    G = flow.G
    king_comp = flow.king_component
    sinks = flow.sinks
    flag = self.flag

    rows = []
    for p in self.pieces:
        if p.place is None: continue
        nid  = p.id
        data = G.nodes[nid]
        net  = data['harvest'] - data['diet']
        max_p = max(p.food.cost * 2, 20)

        rows.append(Tr(
            Td(p.icon()),
            Td(p.name, cls=TextT.medium),
            Td(flag.role_badge(data['is_king'], nid in sinks, net)),
            Td(flag.food_badge(data['harvest'], data['diet'])),
            Td(flag.pantry_bar_badge(p.place.pantry, max_p)),
            Td(flag.net_badge(flow.post_net(nid))),
            Td(flag.link_badge(nid in king_comp)),
        ))

    return Card(
        Table(
            Thead(Tr(Th(""), Th("Name"), Th("Role"),
                     Th("Food"), Th("Pantry"), Th("Net"), Th("Link"))),
            Tbody(*rows),
        ),
        header=(
            H4(self.name),
            P(f"Total surplus: {flow.total_surplus:+d} · Flow: {flow.total_flow:.0f}",
              cls=TextPresets.muted_sm),
        ),
    )


Can we update flowtable to use badges and add the additional to CountryFlag?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

First, new badges on `CountryFlag`:

```python
@patch
def role_badge(self: CountryFlag, is_king, is_sink, net) -> FT:
    if is_king:   emoji, label = "👑", "King"
    elif is_sink: emoji, label = "🍽️", "Sink"
    elif net > 0: emoji, label = "🌾", "Source"
    else:         emoji, label = "⚖️", "Neutral"
    return Span(f"{emoji} {label}",
                style=f"font-size:0.85rem;font-weight:600;color:{self.darkPrimary};")

@patch
def food_badge(self: CountryFlag, harvest, diet) -> FT:
    net = harvest - diet
    arrow = "▲" if net >= 0 else "▼"
    color = "#27ae60" if net >= 0 else "#e74c3c"
    return DivHStacked(
        Span(f"🌾{harvest}", style=f"font-size:0.85rem;color:{self.darkPrimary};"),
        Span(f"🍽️{diet}",   style="font-size:0.85rem;color:#888;"),
        Span(f"{arrow}{abs(net)}", style=f"font-size:0.85rem;font-weight:700;color:{color};"),
        style="gap:6px;"
    )

@patch
def net_badge(self: CountryFlag, post_net: float) -> FT:
    color = "#27ae60" if post_net >= 0 else "#e74c3c"
    return Span(f"{post_net:+.0f}🌾",
                style=f"font-size:1rem;font-weight:700;color:{color};")

@patch
def link_badge(self: CountryFlag, connected: bool) -> FT:
    return Span("✅ Linked" if connected else "❌ Isolated",
                style=f"font-size:0.85rem;color:{'#27ae60' if connected else '#e74c3c'};")

@patch
def pantry_bar_badge(self: CountryFlag, pantry: int, max_p: int) -> FT:
    frac  = max(0.0, min(1.0, pantry / max_p))
    color = self.darkPrimary if pantry > 0 else "#e74c3c"
    return DivVStacked(
        Div(style=(f"background:#ddd;border-radius:3px;width:60px;height:8px;"
                   f"position:relative;")),
        Div(style=(f"background:{color};border-radius:3px;"
                   f"width:{frac*60:.0f}px;height:8px;")),
        Span(f"🧺{pantry}", style=f"font-size:0.85rem;font-weight:700;color:{color};"),
    )
```

Then the updated `flow_table`:

```python
@patch
def flow_table(self: Squad):
    flow = SquadFlow.from_squad(self)
    G = flow.G
    king_comp = flow.king_component
    sinks = flow.sinks
    flag = self.flag

    rows = []
    for p in self.pieces:
        if p.place is None: continue
        nid  = p.id
        data = G.nodes[nid]
        net  = data['harvest'] - data['diet']
        max_p = max(p.food.cost * 2, 20)

        rows.append(Tr(
            Td(p.icon()),
            Td(p.name, cls=TextT.medium),
            Td(flag.role_badge(data['is_king'], nid in sinks, net)),
            Td(flag.food_badge(data['harvest'], data['diet'])),
            Td(flag.pantry_bar_badge(p.place.pantry, max_p)),
            Td(flag.net_badge(flow.post_net(nid))),
            Td(flag.link_badge(nid in king_comp)),
        ))

    return Card(
        Table(
            Thead(Tr(Th(""), Th("Name"), Th("Role"),
                     Th("Food"), Th("Pantry"), Th("Net"), Th("Link"))),
            Tbody(*rows),
        ),
        header=(
            H4(self.name),
            P(f"Total surplus: {flow.total_surplus:+d} · Flow: {flow.total_flow:.0f}",
              cls=TextPresets.muted_sm),
        ),
    )
```

The pantry bar has a small gotcha — the two `Div`s for track + fill need to be nested, not stacked. Let me fix `pantry_bar_badge`:

```python
@patch
def pantry_bar_badge(self: CountryFlag, pantry: int, max_p: int) -> FT:
    frac  = max(0.0, min(1.0, pantry / max_p))
    color = self.darkPrimary if pantry > 0 else "#e74c3c"
    return DivVStacked(
        Div(Div(style=(f"background:{color};border-radius:3px;"
                       f"width:{frac*60:.0f}px;height:8px;")),
            style="background:#ddd;border-radius:3px;width:60px;height:8px;"),
        Span(f"🧺 {pantry}", style=f"font-size:0.85rem;font-weight:700;color:{color};"),
    )
```

I think souce badge should be the same as the cost badge (they are both food) the sink should sugget it is eaten

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good instinct — consistent visual language across the UI. So:

- **Source** → looks like `cost_badge` (🍖 food produced, in flag primary color)
- **Sink** → suggests consumption, maybe 🍽️ with a "being eaten" feel (red/warm tone)

```python
@patch
def role_badge(self: CountryFlag, is_king, is_sink, net, king_pantry=0) -> FT:
    if is_king:
        return DivVStacked(
            Span("👑 King", style=f"font-size:1.1rem;font-weight:700;color:{self.darkPrimary};"),
        )
    elif is_sink:
        return DivVStacked(
            Span("🍽️ Consumed", style="font-size:1.1rem;font-weight:700;color:#e74c3c;"),
            Small("needs feeding", style="color:#e74c3c;font-size:0.75rem;"),
        )
    elif net > 0:
        # Mirror cost_badge style — food being produced
        return DivVStacked(
            Span(f"🍖 +{net}", style=f"font-size:1.1rem;font-weight:700;color:{self.darkPrimary};"),
            Small("✓ producing", style=f"color:#27ae60;font-size:0.75rem;"),
        )
    else:
        return DivVStacked(
            Span("⚖️ Neutral", style="font-size:0.85rem;font-weight:600;color:#888;"),
        )
```

The key changes:
- **Source** mirrors `cost_badge` exactly — 🍖 + amount + affordability note, just green "producing" instead
- **Sink** uses 🍽️ + red "needs feeding" subtext — implies it's consuming
- **King** stays distinct with 👑

why not use pantry_bar?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good catch! We already have `pantry_bar` defined — we should use it. The only snag is it generates SVG fragment coordinates around a `MapCord` center, so we'd need to wrap it in a tiny inline SVG. Something like:

```python
def _pantry_svg(piece, flow, nid, radius=20):
    cx, cy = radius + 4, radius + 4
    center = MapCord(cx, cy)
    w = radius * 2 + 8
    h = radius * 0.6 + cy + 4
    bar = pantry_bar(piece, flow, nid, center, radius)
    return f'<svg width="{w:.0f}" height="{h:.0f}">{bar}</svg>'
```

Then in `flow_table` replace the inline bar with:

```python
Td(NotStr(_pantry_svg(p, flow, nid)))
```

But honestly — looking at `pantry_bar` again, it's designed for **map overlay** rendering where the bar sits below the piece icon at a given pixel position. For a table it's a bit overbuilt.

Two options:
1. **Wrap it in a mini SVG** as above — reuses the code exactly
2. **Extract the bar logic** into a shared helper that both `pantry_bar` and `flow_table` call

Which feels cleaner to you?

In [ ]:
#| export
@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    node_scale: float = 0.9
    on_click: Callable[[Piece], dict] | None = None
    label_fn: Callable[[Piece, SquadFlow, str, MapCord], str] | None = None
    
    
    def __ft__(self):
        return self._render()

I feel the SquadFlowDiagram should be refactored. That is one large function. Are there other things we should add?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yeah, `_render` is doing at least 5 distinct jobs. Here's how I'd break it down:

**Natural extraction points:**
1. **`_layout(G)`** → `dict[nid, (x,y)]` — the kamada-kawai + scaling to SVG coords
2. **`_render_edges(flow, svg_pos, builder)`** → edge SVG string
3. **`_render_nodes(flow, svg_pos, builder)`** → node SVG string  
4. **`_render_footer(flow)`** → the summary `P(...)` element
5. **`_render(flow)`** stays as the thin orchestrator

**Things to consider adding:**

- **Legend** — what the colors/dashes/rings mean (red ring = bottleneck, green = surplus, etc.)
- **Disconnected pieces** — they're currently invisible since they have no edges. Could show them in a "graveyard" sidebar or greyed-out zone
- **Flow animation** — `stroke-dashoffset` CSS animation along edges to show direction
- **Tooltips** — `<title>` elements on nodes with full stats
- **Diff mode** — highlight what changed between two `SquadFlow` snapshots (new edges, lost connections, pantry changes)

Which of these feel most useful to tackle first? And do you want to start with the refactor (extracting the methods) or add features on top of the current structure?

In [ ]:
#| export
@patch
def _layout(self:SquadFlowDiagram, G) -> dict:
        """Compute SVG positions for each node using kamada-kawai."""
        node_r = 22.5 * self.node_scale
        pad = node_r * 2.5
        inner_w = self.width - pad * 2
        inner_h = self.height - pad * 2
        
        if len(G) == 1:
            pos = {n: np.array([0.0, 0.0]) for n in G.nodes}
        else:
            pos = nx.kamada_kawai_layout(G)
        
        xs = [p[0] for p in pos.values()]
        ys = [p[1] for p in pos.values()]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        x_range = x_max - x_min or 1.0
        y_range = y_max - y_min or 1.0
        
        return {
            nid: MapCord(pad + (p[0] - x_min) / x_range * inner_w,
                  pad + (p[1] - y_min) / y_range * inner_h)
            for nid, p in pos.items()
        }


In [ ]:
#| export
@patch
def _render_edges(self: SquadFlowDiagram, flow, svg_pos, builder) -> str:
    r = SquadFlowRenderer(flag=self.squad.flag, node_r=22.5 * self.node_scale)
    return r.all_edges(flow, svg_pos, builder, shrink=22.5 * self.node_scale + 8)




!cat ../../HexMagic/game/flag.py

### Render Node

In [ ]:
#| export
@patch
def _render_node(self: SquadFlowDiagram, nid, data, flow, svg_pos, builder) -> str:
    r = SquadFlowRenderer(flag=self.squad.flag, node_r=22.5 * self.node_scale)
    center = svg_pos[nid]
    piece = data['piece']

    # Piece chess icon from flag
    svg_str, pat_def = self.squad.flag.piece_svg(
        piece.type, center,
        scale=self.node_scale, size='board',
        piece_id=f"flow_{nid}_{piece.type.value}")
    if pat_def is not None:
        builder.add_definition(pat_def)

    # Death marker if starving
    post_net = flow.post_net(nid)
    glyphs = DiagramGlyphs(self.squad.flag, size=22.5 * self.node_scale)
    glyphs.register_styles(builder)
    death = ""
    if post_net < 0 and not data['is_king']:
        ring_r = 22.5 * self.node_scale + 6
        death = glyphs.death_marker(
            MapCord(center.x - ring_r * 0.7, center.y - ring_r * 0.7),
            size=22.5 * self.node_scale * 0.4)

    hx_attrs = ""
    if self.on_click:
        hx_attrs = ' '.join(f'{k}="{v}"' for k, v in self.on_click(piece).items())

    parts = []
    parts = [f'<g style="cursor:pointer" {hx_attrs}>' if hx_attrs else '<g>',
             r.node_ring_fragment(nid, data, flow, center, builder),
             svg_str,
             death,
             #r.node_food_pip(nid, flow, center, builder),
             #r.node_labels(piece, flow, nid, center),
             '</g>']

    #parts = []

    label = (self.label_fn(piece, flow, nid, center) 
    if self.label_fn else r.node_labels(piece, flow, nid, center))
    parts.append(label)
    return '\n'.join(p for p in parts if p)


In [ ]:
#| export
@patch
def _render_nodes(self:SquadFlowDiagram, flow, svg_pos, builder) -> str:
    """Render all nodes."""
    return '\n'.join(
        self._render_node(nid, data, flow, svg_pos, builder)
        for nid, data in flow.G.nodes(data=True)
    )

In [ ]:
#| export
@patch
def _render_footer(self: SquadFlowDiagram, flow) -> FT:
    """Summary stats bar using flag badges."""
    flag = self.squad.flag
    king = next((p for p in self.squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    king_pantry = king.place.pantry if king else 0
    bn_count = len(flow.bottlenecks)
    target = getattr(self.squad, '_target', None)
    target_str = target.label if target else "none"

    bn_color = "#e74c3c" if bn_count > 0 else "#27ae60"

    return DivHStacked(
        flag.pantry_badge(king_pantry),
        flag.net_badge(flow.total_surplus),
        Span(f"🔄 {flow.total_flow:.0f}", 
             style=f"font-size:1.1rem;font-weight:700;color:{flag.darkPrimary};"),
        Span(f"⚠️ {bn_count} bottleneck{'s' if bn_count != 1 else ''}",
             style=f"font-size:0.85rem;font-weight:600;color:{bn_color};"),
        Span(f"🎯 {target_str}",
             style=f"font-size:0.85rem;color:{flag.darkPrimary};"),
        style="gap:16px; align-items:center; flex-wrap:wrap; padding:4px 8px;"
    )


_render_footer should use badges

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
@patch
def _render_footer(self: SquadFlowDiagram, flow) -> FT:
    """Summary stats bar using flag badges."""
    flag = self.squad.flag
    king = next((p for p in self.squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    king_pantry = king.place.pantry if king else 0
    bn_count = len(flow.bottlenecks)
    target = getattr(self.squad, '_target', None)
    target_str = target.label if target else "none"

    bn_color = "#e74c3c" if bn_count > 0 else "#27ae60"

    return DivHStacked(
        flag.pantry_badge(king_pantry),
        flag.net_badge(flow.total_surplus),
        Span(f"🔄 {flow.total_flow:.0f}", 
             style=f"font-size:1.1rem;font-weight:700;color:{flag.darkPrimary};"),
        Span(f"⚠️ {bn_count} bottleneck{'s' if bn_count != 1 else ''}",
             style=f"font-size:0.85rem;font-weight:600;color:{bn_color};"),
        Span(f"🎯 {target_str}",
             style=f"font-size:0.85rem;color:{flag.darkPrimary};"),
        style="gap:16px; align-items:center; flex-wrap:wrap; padding:4px 8px;"
    )
```

Key changes:
- `pantry_badge` for king's pantry — consistent with the table
- `net_badge` for surplus — same green/red coloring as the table's Net column
- Bottleneck count turns red when there are any, green when clear
- `DivHStacked` with gap so it flows nicely as a footer row

In [ ]:
#| export
@patch
def _render(self:SquadFlowDiagram, flow: SquadFlow = None):
    if flow is None:
        flow = SquadFlow.from_squad(self.squad)
    G = flow.G
    
    if len(G) < 1:
        return Div(P(f"{self.squad.name}: no placed pieces"))
    
    flag = self.squad.flag
    placed_count = sum(1 for p in self.squad.pieces if p.place is not None)
    
    builder = SVGBuilder()
    builder.width = self.width
    builder.height = self.height
    
    svg_pos = self._layout(G)
    builder.adjust("edges", self._render_edges(flow, svg_pos, builder))
    builder.adjust("nodes", self._render_nodes(flow, svg_pos, builder))
    
    return Card(
        NotStr(builder.xml()),
        footer=self._render_footer(flow),
        header=DivFullySpaced(
            Div(H4(self.squad.name),
                P(f"{flag.name} · {placed_count} placed")),
        ),
    )

In [ ]:
??_pantry_bar

In [ ]:
#from HexMagic.magic.food import LIFESPAN  # if it's exported there, otherwise adjust

def make_diagram(types, connections, **kw):
    """Helper: wire a squad and wrap in SquadFlowDiagram."""
    sq = wire_squad(types, connections)
    return SquadFlowDiagram(squad=sq, parts=myStuff, **kw)

# --- _layout tests ---

def test_layout_positions_in_bounds():
    """All node positions should be within the SVG dimensions."""
    diag = make_diagram(
        [PieceType.KING, PieceType.PAWN, PieceType.PAWN],
        {0: [1, 2], 1: [0, 2], 2: [0, 1]})
    flow = SquadFlow.from_squad(diag.squad)
    pos = diag._layout(flow.G)
    
    assert len(pos) == 3
    for nid, cord in pos.items():
        x = cord.x
        y = cord.y
        assert 0 <= x <= diag.width, f"{nid} x={x} out of bounds"
        assert 0 <= y <= diag.height, f"{nid} y={y} out of bounds"
    print("✅ layout positions in bounds")

def test_layout_single_node():
    """Single node should still produce a valid position."""
    diag = make_diagram([PieceType.KING], {0: []})
    flow = SquadFlow.from_squad(diag.squad)
    pos = diag._layout(flow.G)
    assert len(pos) == 1
    print("✅ layout single node")

# --- _render_edges tests ---

def test_edges_svg_contains_lines():
    """Edge rendering should produce <path> elements (MapPath.with_arrowhead)."""
    diag = make_diagram(
        [PieceType.KING, PieceType.PAWN],
        {0: [1], 1: [0]})
    flow = SquadFlow.from_squad(diag.squad)
    svg_pos = diag._layout(flow.G)
    builder = SVGBuilder()
    builder.width, builder.height = diag.width, diag.height
    
    edge_svg = diag._render_edges(flow, svg_pos, builder)
    assert "<path" in edge_svg
    print("✅ edges contain <path> elements")

def test_edges_no_crash_on_disconnected():
    """Disconnected pieces should produce no edges to them."""
    diag = make_diagram(
        [PieceType.KING, PieceType.PAWN, PieceType.PAWN],
        {0: [1], 1: [0], 2: []})
    flow = SquadFlow.from_squad(diag.squad)
    svg_pos = diag._layout(flow.G)
    builder = SVGBuilder()
    builder.width, builder.height = diag.width, diag.height
    
    edge_svg = diag._render_edges(flow, svg_pos, builder)
    # 1 undirected edge → 1 <path> from drawPolygon + 1 arrowhead polygon
    assert edge_svg.count("<path") >= 1
    assert edge_svg.count("<path") < 4  # not drawing extra edges
    print("✅ edges handle disconnected pieces")


# --- _render_footer tests ---

def test_footer_shows_bottleneck_count():
    """Linear chain should report 1 bottleneck."""
    diag = make_diagram(
        [PieceType.PAWN, PieceType.PAWN, PieceType.KING],
        {0: [1], 1: [0, 2], 2: [1]})
    flow = SquadFlow.from_squad(diag.squad)
    footer = diag._render_footer(flow)
    # footer is a P() element — convert to string to check
    footer_str = str(to_xml(footer))
    assert "1 bottleneck" in footer_str
    print("✅ footer bottleneck count")

def test_footer_fully_connected_no_bottlenecks():
    diag = make_diagram(
        [PieceType.KING, PieceType.PAWN, PieceType.PAWN],
        {0: [1, 2], 1: [0, 2], 2: [0, 1]})
    flow = SquadFlow.from_squad(diag.squad)
    footer_str = str(to_xml(diag._render_footer(flow)))
    assert "0 bottlenecks" in footer_str
    print("✅ footer no bottlenecks")

# --- Full _render integration ---

def test_render_produces_card():
    """Full render should return a Card-like structure without crashing."""
    diag = make_diagram(
        [PieceType.KING, PieceType.PAWN, PieceType.PAWN],
        {0: [1], 1: [0, 2], 2: [1]})
    result = diag._render()
    result_str = str(to_xml(result))
    assert "<article" in result_str  # Card renders as <article>
    assert diag.squad.name in result_str
    print("✅ full render produces card")

def test_render_empty_squad():
    """Squad with no placed pieces should return a simple message."""
    flag = CountryFlag.seaborn("husl", 1)[0]
    sq = Squad(pieces=[], name="Empty", flag=flag)
    diag = SquadFlowDiagram(squad=sq, parts=myStuff)
    result_str = str(to_xml(diag._render()))
    assert "no placed pieces" in result_str
    print("✅ render empty squad")

# Run all
test_layout_positions_in_bounds()
test_layout_single_node()
test_edges_svg_contains_lines()
test_edges_no_crash_on_disconnected()
test_footer_shows_bottleneck_count()
test_footer_fully_connected_no_bottlenecks()
test_render_produces_card()
test_render_empty_squad()


Can you make a nice demo of SquadFlowDiagram

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
def squad_flow_demo():
    """Demo showing SquadFlowDiagram across three topology scenarios."""

    scenarios = [
        ("Chain ⛓️",
         [PieceType.PAWN, PieceType.PAWN, PieceType.PAWN, PieceType.KING],
         {0: [1], 1: [0, 2], 2: [1, 3], 3: [2]},
         "Linear supply chain — p1 and p2 are bottlenecks"),

        ("Star ⭐",
         [PieceType.KING, PieceType.PAWN, PieceType.PAWN, PieceType.PAWN],
         {0: [1, 2, 3], 1: [0], 2: [0], 3: [0]},
         "All pieces report directly to king — king is the bottleneck"),

        ("Mesh 🕸️",
         [PieceType.KING, PieceType.PAWN, PieceType.PAWN, PieceType.PAWN],
         {0: [1, 2], 1: [0, 2, 3], 2: [0, 1, 3], 3: [1, 2]},
         "Redundant connections — no single point of failure"),
    ]

    cards = []
    for title, types, connections, caption in scenarios:
        sq = wire_squad(types, connections)
        diag = SquadFlowDiagram(squad=sq, width=400, height=300)
        cards.append(
            Card(
                diag._render(),
                header=DivLAligned(H4(title), P(caption, cls=TextPresets.muted_sm)),
            )
        )

    return Div(
        H2("SquadFlowDiagram Demo"),
        P("Three network topologies showing food flow, bottlenecks, and connectivity.",
          cls=TextPresets.muted_sm),
        Grid(*cards, cols_sm=1, cols_md=3),
    )

show(squad_flow_demo())
```

This shows three classic topologies side by side:

| Scenario | What to look for |
|---|---|
| **Chain** | Red dashed rings on middle pawns — they're articulation points |
| **Star** | King node lights up as the single hub |
| **Mesh** | No bottleneck rings — redundant paths protect the network |

The food pips and death markers on each node will vary since `FakePlace` uses `harvest=1.0` for everyone — if you want more interesting food gradients you can add a `harvests` param to `wire_squad` to vary it per piece.

#| export
@patch
def _render_node(self: SquadFlowDiagram, nid, data, flow, svg_pos, builder) -> str:
    flag = self.squad.flag
    node_r = 22.5 * self.node_scale
    piece = data['piece']
    center: MapCord = svg_pos[nid]

    post_net = flow.post_net(nid)
    tenure   = piece.place.tenure if piece.place else 0
    life     = piece.food.lifespan
    pantry   = piece.place.pantry if piece.place else 0
    turns_left = max(0, life - tenure) if life < 999 else -1

    # Use DiagramGlyphs for all geometry
    glyphs = DiagramGlyphs(flag, size=node_r)
    glyphs.register_styles(builder)

    ring_r = node_r + 6
    parts  = []

    # HTMX group wrapper
    hx_attrs = ""
    if self.on_click:
        hx_attrs = ' '.join(f'{k}="{v}"' for k, v in self.on_click(piece).items())
    parts.append(f'<g style="cursor:pointer" {hx_attrs}>' if hx_attrs else '<g>')

    # Bottleneck dashed outer ring via DiagramGlyphs
    if nid in flow.bottlenecks:
        danger = glyphs.danger_style()
        builder.add_style(danger)
        parts.append(glyphs.ring(center, ring_r + 4, danger))

    # Main ring — gold/green/red via outline_style override
    ring_color = ("#FFD700" if data['is_king']
                  else "#4CAF50" if post_net >= 0
                  else "#E53935")
    ring_style = StyleCSS(f"node_ring_{nid[:6]}",
                          fill="white", stroke=ring_color,
                          stroke_width=3.5 if data['is_king'] else 2.5)
    builder.add_style(ring_style)
    parts.append(glyphs.ring(center, ring_r, ring_style))

    # Chess piece from flag
    svg_str, pat_def = flag.piece_svg(
        piece.type, center,
        scale=self.node_scale, size='board',
        piece_id=f"flow_{nid}_{piece.type.value}")
    if pat_def is not None:
        builder.add_definition(pat_def)
    parts.append(svg_str)

    # Food pip: maps post_net to 0–1 range, colored red→green
    frac = min(1.0, max(0.0, (post_net + 10) / 20))
    pip_center = MapCord(center.x + ring_r * 0.7, center.y - ring_r * 0.7)
    parts.append(glyphs.food_pip(pip_center, frac, max_r=8, min_r=4))

    # Death marker if dying
    if post_net < 0 and not data['is_king']:
        parts.append(glyphs.death_marker(
            MapCord(center.x - ring_r * 0.7, center.y - ring_r * 0.7),
            size=node_r * 0.4))

    # Name + stats labels
    name_y  = center.y + ring_r + 14
    turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
    parts.append(
        f'<text x="{center.x:.1f}" y="{name_y:.1f}" text-anchor="middle" '
        f'font-size="14"  fill="{flag.darkPrimary}">{piece.name}</text>')
    parts.append(
        f'<text x="{center.x:.1f}" y="{name_y + 13:.1f}" text-anchor="middle" font-size="12">'
    )
    if  post_net < 0: 
        parts.append(f'<tspan fill="#E53935">{post_net:+.0f} </tspan>')

    parts.append(   
        f'  <tspan fill="#888">📦{pantry}</tspan>'
        f'  <tspan fill="#666">{turns_str}</tspan></text>')

    parts.append('</g>')
    return '\n'.join(parts)


What is the black dot in the upper right corner on these nodes?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's the `food_pip` rendered at `pip_center = MapCord(center.x + ring_r * 0.7, center.y - ring_r * 0.7)`.

It's appearing dark/black because in the fake setup `harvest=1.0` but `diet` from `piece.food` is likely higher, making `post_net` very negative, which pushes `frac` close to `0` — and the red→green lerp at `frac=0` gives `#e74c3c` which at a small radius (`min_r=4`) looks almost black.

Two easy fixes — pick one:

**Option A: clamp frac to a visible range** so it's always at least a clear red:
```python
frac = min(1.0, max(0.15, (post_net + 10) / 20))
```

**Option B: add a white stroke so it reads clearly at any color:**
```python
pip_style = StyleCSS(f"pip_{int(frac*100)}",
    fill=color, stroke="white", stroke_width=1.5, opacity=opacity)
```
(This change would go in `food_pip` on `DiagramGlyphs`.)

**Option C: use a larger `min_r`** so it's big enough to read as colored:
```python
parts.append(glyphs.food_pip(pip_center, frac, max_r=10, min_r=6))
```

Option B is probably the most robust since it makes the pip readable regardless of color or size.

In [ ]:
def squad_flow_demo():
    """Demo showing SquadFlowDiagram across three topology scenarios."""

    scenarios = [
        ("Chain ⛓️",
         [PieceType.PAWN, PieceType.PAWN, PieceType.PAWN, PieceType.KING],
         {0: [1], 1: [0, 2], 2: [1, 3], 3: [2]},
         "Linear supply chain — p1 and p2 are bottlenecks"),

        ("Star ⭐",
         [PieceType.KING, PieceType.PAWN, PieceType.PAWN, PieceType.PAWN],
         {0: [1, 2, 3], 1: [0], 2: [0], 3: [0]},
         "All pieces report directly to king — king is the bottleneck"),

        ("Mesh 🕸️",
         [PieceType.KING, PieceType.PAWN, PieceType.PAWN, PieceType.PAWN],
         {0: [1, 2], 1: [0, 2, 3], 2: [0, 1, 3], 3: [1, 2]},
         "Redundant connections — no single point of failure"),
    ]

    cards = []
    for title, types, connections, caption in scenarios:
        sq = wire_squad(types, connections)
        diag = SquadFlowDiagram(squad=sq, parts=myStuff, width=400, height=300)
        
        
        #,label_fn=lambda p, f, n, c: f'<text x="{c.x}" y="{c.y+40}" text-anchor="middle">{p.name}</text>')

        cards.append(
            Card(
                diag._render(),
                header=DivLAligned(H4(title), P(caption, cls=TextPresets.muted_sm)),
            )
        )

    return Div(
        H2("SquadFlowDiagram Demo"),
        P("Three network topologies showing food flow, bottlenecks, and connectivity.",
          cls=TextPresets.muted_sm),
        Grid(*cards, cols_sm=1, cols_md=3),
    )

show(squad_flow_demo())


how do i fix?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The class is missing `@dataclass`. Looking at the definition:

```python
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    ...
```

It has dataclass-style field annotations but no `@dataclass` decorator, so Python treats them as plain class variables and generates no `__init__`. Fix:

```python
@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    node_scale: float = 0.9
    on_click: Callable[[Piece], dict] | None = None
    label_fn: Callable[[Piece, SquadFlow, str, MapCord], str] | None = None

    def __ft__(self):
        return self._render()
```

SquadFlowDiagram should take parameters for its node label

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good idea — right now `node_labels` is hardcoded to show name + net + pantry + turns. You could make it configurable via a callable or a set of flags.

Two design options:

**Option A: `label_fn` callable** — most flexible:
```python
@dataclass
class SquadFlowDiagram:
    squad: Squad
    parts: GameParts
    width: int = 600
    height: int = 400
    node_scale: float = 0.9
    on_click: Callable[[Piece], dict] | None = None
    label_fn: Callable[[Piece, SquadFlow, str, MapCord], str] | None = None
```

Then in `_render_node`, replace `r.node_labels(...)` with:
```python
label = (self.label_fn(piece, flow, nid, center) 
         if self.label_fn else r.node_labels(piece, flow, nid, center))
parts.append(label)
```

Usage:
```python
# Name only
diag = SquadFlowDiagram(..., label_fn=lambda p, f, n, c: 
    f'<text x="{c.x}" y="{c.y+40}" text-anchor="middle">{p.name}</text>')
```

**Option B: `label_fields` set** — simpler to use:
```python
label_fields: set[str] = field(default_factory=lambda: {'name','net','pantry','turns'})
```

Then `node_labels` checks `if 'pantry' in self.label_fields` etc.

Option A is more powerful — you can pass in any SVG string. Option B is easier for quick tweaks like "just name + net". Which suits your use case better?

### overlay

In [ ]:
#| export
def FoodNetworkOverlay(
    focus_squad: Squad | None = None,
    edge_opacity: float = 0.6,
    bottleneck_r_mult: float = 1.4,
    **kw
) -> OverlaySpec:

    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts = []

        for squad in ctx.squads:
            flow = SquadFlow.from_squad(squad)
            G = flow.G
            if len(G) < 1: continue

            flag = squad.flag
            is_focused = focus_squad is None or squad is focus_squad
            r = SquadFlowRenderer(flag=flag,
                                  node_r=grid.radius * 0.5,
                                  edge_opacity=edge_opacity)

            # ── Build svg_pos from hex grid centers ──────────────────
            svg_pos = {}
            for nid, data in G.nodes(data=True):
                idx = data['hex_idx']
                fis = [fi for fi in ctx.fine_indices(idx) if 0 <= fi < N]
                if fis:
                    svg_pos[nid] = grid.hexes[fis[0]].center

            missing = [n for n in G.nodes if n not in svg_pos]
            for n in missing:
                G.remove_node(n)  # off-grid pieces: skip silently

            if not svg_pos: continue

            # ── Edges ────────────────────────────────────────────────
            if is_focused and len(G) > 1:
                parts.append(r.all_edges(flow, svg_pos, ctx.builder,
                                         shrink=grid.radius * 0.4))

            # ── Target arrow (overlay-only, no equivalent in diagram) ─
            if is_focused:
                parts.append(_target_arrow(squad, G, svg_pos, grid, flag))

            # ── Nodes ────────────────────────────────────────────────
            for nid, data in G.nodes(data=True):
                if nid not in svg_pos: continue
                center = svg_pos[nid]
                piece  = data['piece']

                parts.append(r.node_ring_fragment(nid, data, flow, center, ctx.builder))
                parts.append(r.node_food_pip(nid, flow, center, ctx.builder))

                if is_focused:
                    parts.append(_pantry_bar(piece, flow, nid, center, grid))
                    parts.append(r.node_labels(piece, flow, nid, center))

        return '\n'.join(p for p in parts if p)

    return OverlaySpec("food_network", render, requires={'squads'}, priority=63)


def _target_arrow(squad, G, svg_pos, grid, flag) -> str:
    """Target direction indicator — overlay-specific, no diagram equivalent."""
    target = getattr(squad, '_target', None)
    if not target or target.hex_idx < 0: return ""
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id or king_id not in svg_pos: return ""

    kc = svg_pos[king_id]
    tc = grid.hexes[target.hex_idx].center
    dx, dy = tc.x - kc.x, tc.y - kc.y
    dist = math.sqrt(dx*dx + dy*dy)
    if dist < 1: return ""

    ux, uy = dx / dist, dy / dist
    arrow_len = grid.radius * 2.5
    ax, ay = kc.x + ux * arrow_len, kc.y + uy * arrow_len
    icon = "⚔️" if target.type == TargetType.KING else "🌾"
    sw = max(3, grid.radius * 0.15)
    fs = max(12, grid.radius * 0.5)

    return (f'<line x1="{kc.x:.1f}" y1="{kc.y:.1f}" x2="{ax:.1f}" y2="{ay:.1f}" '
            f'stroke="{flag.primary}" stroke-opacity="0.4" stroke-width="{sw:.1f}" '
            f'stroke-linecap="round" stroke-dasharray="8,4"/>'
            f'<text x="{ax:.1f}" y="{ay:.1f}" text-anchor="middle" '
            f'font-size="{fs:.0f}" dominant-baseline="central">{icon}</text>')


def _pantry_bar(piece, flow, nid, center: MapCord, grid) -> str:
    """Pantry fill bar — map overlay only (diagram uses footer instead)."""
    pantry   = piece.place.pantry if piece.place else 0
    post_net = flow.post_net(nid)
    bar_w    = grid.radius * 0.8
    bar_h    = max(3, grid.radius * 0.1)
    bar_x    = center.x - bar_w / 2
    bar_y    = center.y + grid.radius * 0.5 * 2.0
    max_p    = max(piece.food.cost * 2, 20)
    frac     = max(0.0, min(1.0, pantry / max_p))
    color    = "#4CAF50" if pantry > 0 else "#E53935"

    return (f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" width="{bar_w:.1f}" '
            f'height="{bar_h:.1f}" rx="1.5" fill="#ddd" opacity="0.6"/>'
            f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" width="{bar_w*frac:.1f}" '
            f'height="{bar_h:.1f}" rx="1.5" fill="{color}" opacity="0.8"/>')


foodNetworkOverlay can be refactored like the diagram do you need to see mapCord and mapPath and the glyphs to do so?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I can see enough from the existing code to suggest the split. `FoodNetworkOverlay.render` is doing five distinct jobs:

1. **`_edge_svg(flow, G, ctx, flag, arrow_id)`** → edge lines + flow labels
2. **`_target_arrow_svg(squad, G, ctx, flag)`** → the target direction indicator
3. **`_node_svg(nid, data, flow, ctx, glyphs, is_focused)`** → per-node glyph + pantry bar + label
4. **`_register_arrow_marker(ctx, flag)`** → the `SVGDef` marker setup (currently buried in the edge loop)
5. **`render(ctx)`** stays as the thin orchestrator loop over squads

The parallel to `SquadFlowDiagram` would be:

| Diagram | Overlay |
|---|---|
| `_layout` | `_register_arrow_marker` |
| `_render_edges` | `_edge_svg` |
| `_render_node` | `_node_svg` |
| `_render_footer` | `_target_arrow_svg` |
| `_render` | `render` |

The trickiest part is that `_edge_svg` currently iterates `ctx.fine_indices` in a double loop — worth extracting a small helper `_hex_center_pair(ctx, src_idx, dst_idx)` that yields `(c1, c2)` pairs to keep each method clean.

Do you want me to write the refactored version?

should FoodNetworkOverlay contain a squadflowdiagram? is there overlay between the classes we should extract?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good design question. Looking at both classes, there's clear overlap:

**Shared logic right now:**
- Both call `SquadFlow.from_squad(squad)` to get `flow`
- Both iterate `flow.G.nodes` and `flow.bottlenecks`
- Both do the same edge color/width logic (green=flow, red=bridge, dim=sight-only)
- Both render a per-node "post_net → color" decision
- Both use `DiagramGlyphs` for bottleneck rings and food pips

**The relationship question:**
`SquadFlowDiagram` is a *standalone SVG panel* while `FoodNetworkOverlay` is a *map layer*. They shouldn't contain each other, but they should share a common **renderer strategy**. The natural extraction is:

```
SquadFlowRenderer          # pure logic: given flow+positions → SVG fragments
    .edge_fragments(flow, pos_pairs) 
    .node_fragment(nid, data, flow, center, glyphs)
    .edge_color(flow, u, v)  # the shared color/width decision

SquadFlowDiagram           # uses kamada-kawai positions
    → delegates to SquadFlowRenderer

FoodNetworkOverlay         # uses hex grid positions  
    → delegates to SquadFlowRenderer
```

The key insight is that the *only* difference between the two is **where the node positions come from** — everything else (color rules, bottleneck rings, flow labels) is identical and currently duplicated.

Worth extracting `SquadFlowRenderer` as a separate class?

yes lest extract SquadFlowRenderer

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the extracted `SquadFlowRenderer`:

```python
@dataclass
class SquadFlowRenderer:
    """Pure SVG fragment generator for squad food flow.
    Shared by SquadFlowDiagram (panel) and FoodNetworkOverlay (map layer).
    Position source is injected — caller decides where nodes live."""
    
    flag: 'CountryFlag'
    node_r: float = 20.0
    edge_opacity: float = 0.6

    # ── Edge helpers ────────────────────────────────────────────────

    def edge_color_width(self, flow: SquadFlow, u, v) -> tuple[str, float, str]:
        """Returns (color, stroke_width, dash) for an undirected edge."""
        net = abs(flow.edge_flow(u, v))
        is_bridge = u in flow.bottlenecks or v in flow.bottlenecks

        if net > 0:
            return "#4CAF50", 1.5 + (net / flow.max_flow) * 4, ""
        elif is_bridge:
            return "#E53935", 1.5, "4,3"
        else:
            return self.flag.darkPrimary, 1.0, "4,3"

    def edge_fragment(self, flow: SquadFlow,
                      u, v,
                      p1: MapCord, p2: MapCord,
                      builder,
                      shrink: float = 30.0) -> str:
        """SVG for one undirected edge: arrow + optional flow label."""
        dist = p1.distance(p2)
        if dist < 1: return ""

        net = flow.edge_flow(u, v)
        total = abs(net)
        src_p, dst_p = (p1, p2) if net >= 0 else (p2, p1)

        dx, dy = (dst_p.x - src_p.x) / dist, (dst_p.y - src_p.y) / dist
        start = MapCord(src_p.x + dx * shrink, src_p.y + dy * shrink)
        end   = MapCord(dst_p.x - dx * shrink, dst_p.y - dy * shrink)

        color, sw, dash = self.edge_color_width(flow, u, v)
        opacity = self.edge_opacity if total > 0 else self.edge_opacity * 0.4

        style = StyleCSS(f"sfr_edge_{u[:4]}_{v[:4]}",
                         stroke=color, stroke_width=sw,
                         fill=color, opacity=opacity,
                         **{"stroke-dasharray": dash} if dash else {})
        builder.add_style(style)

        parts = [MapPath([start, end], style).with_arrowhead(arrow_size=10, arrow_angle=25)]

        if total > 0:
            mid = start.midPoint(end)
            px, py = -dy * 12, dx * 12
            parts.append(
                f'<text x="{mid.x + px:.1f}" y="{mid.y + py:.1f}" '
                f'text-anchor="middle" font-size="10" font-weight="bold" '
                f'fill="#4CAF50" opacity="0.8">{total:.0f}🌾</text>')

        return '\n'.join(str(p) for p in parts)

    def all_edges(self, flow: SquadFlow,
                  svg_pos: dict[str, MapCord],
                  builder,
                  shrink: float = 30.0) -> str:
        """All edges, deduped."""
        seen = set()
        parts = []
        for u, v in flow.G.edges():
            key = tuple(sorted([u, v]))
            if key in seen: continue
            seen.add(key)
            parts.append(self.edge_fragment(flow, u, v,
                                            svg_pos[u], svg_pos[v],
                                            builder, shrink))
        return '\n'.join(parts)

    # ── Node helpers ─────────────────────────────────────────────────

    def node_ring_fragment(self, nid, data, flow: SquadFlow,
                           center: MapCord, builder) -> str:
        """Bottleneck outer ring + colored inner ring."""
        glyphs = DiagramGlyphs(self.flag, size=self.node_r)
        glyphs.register_styles(builder)
        ring_r = self.node_r + 6
        parts = []

        if nid in flow.bottlenecks:
            danger = glyphs.danger_style()
            builder.add_style(danger)
            parts.append(glyphs.ring(center, ring_r + 4, danger))

        post_net = flow.post_net(nid)
        ring_color = ("#FFD700" if data['is_king']
                      else "#4CAF50" if post_net >= 0
                      else "#E53935")
        ring_style = StyleCSS(f"sfr_ring_{nid[:6]}",
                              fill="white", stroke=ring_color,
                              stroke_width=3.5 if data['is_king'] else 2.5)
        builder.add_style(ring_style)
        parts.append(glyphs.ring(center, ring_r, ring_style))
        return '\n'.join(parts)

    def node_food_pip(self, nid, flow: SquadFlow,
                      center: MapCord, builder) -> str:
        """Food pip in upper-right of node."""
        glyphs = DiagramGlyphs(self.flag, size=self.node_r)
        glyphs.register_styles(builder)
        ring_r = self.node_r + 6
        post_net = flow.post_net(nid)
        frac = min(1.0, max(0.0, (post_net + 10) / 20))
        pip_center = MapCord(center.x + ring_r * 0.7, center.y - ring_r * 0.7)
        return glyphs.food_pip(pip_center, frac, max_r=8, min_r=4)

    def node_labels(self, piece, flow: SquadFlow, nid,
                    center: MapCord) -> str:
        """Name + stats text below node."""
        ring_r = self.node_r + 6
        post_net = flow.post_net(nid)
        pantry = piece.place.pantry if piece.place else 0
        tenure = piece.place.tenure if piece.place else 0
        life = piece.food.lifespan
        turns_left = max(0, life - tenure) if life < 999 else -1
        turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
        name_y = center.y + ring_r + 14
        net_color = "#4CAF50" if post_net >= 0 else "#E53935"

        parts = [
            f'<text x="{center.x:.1f}" y="{name_y:.1f}" '
            f'text-anchor="middle" font-size="14" fill="{self.flag.darkPrimary}">'
            f'{piece.name}</text>',
            f'<text x="{center.x:.1f}" y="{name_y+13:.1f}" '
            f'text-anchor="middle" font-size="12">'
            f'<tspan fill="{net_color}">{post_net:+.0f}🌾</tspan> '
            f'<tspan fill="#888">📦{pantry}</tspan> '
            f'<tspan fill="#666">{turns_str}</tspan></text>'
        ]
        return '\n'.join(parts)
```

Then `SquadFlowDiagram._render_edges` and `_render_node` become thin wrappers:

```python
@patch
def _render_edges(self: SquadFlowDiagram, flow, svg_pos, builder) -> str:
    r = SquadFlowRenderer(flag=self.squad.flag, node_r=22.5 * self.node_scale)
    return r.all_edges(flow, svg_pos, builder, shrink=22.5 * self.node_scale + 8)

@patch
def _render_node(self: SquadFlowDiagram, nid, data, flow, svg_pos, builder) -> str:
    r = SquadFlowRenderer(flag=self.squad.flag, node_r=22.5 * self.node_scale)
    center = svg_pos[nid]
    piece = data['piece']
    parts = ['<g>',
             r.node_ring_fragment(nid, data, flow, center, builder),
             # piece SVG (stays in diagram — flag-specific)
             r.node_food_pip(nid, flow, center, builder),
             r.node_labels(piece, flow, nid, center),
             '</g>']
    return '\n'.join(parts)
```

And `FoodNetworkOverlay` replaces its duplicate edge/node logic with the same `SquadFlowRenderer` calls, just passing hex grid centers instead of kamada-kawai positions.

Want to wire up the overlay side next?

@dataclass
class SquadFlowRenderer:
    """Pure SVG fragment generator for squad food flow.
    Shared by SquadFlowDiagram (panel) and FoodNetworkOverlay (map layer).
    Position source is injected — caller decides where nodes live."""
    
    flag: 'CountryFlag'
    node_r: float = 20.0
    edge_opacity: float = 0.6

    # ── Edge helpers ────────────────────────────────────────────────

    def edge_color_width(self, flow: SquadFlow, u, v) -> tuple[str, float, str]:
        """Returns (color, stroke_width, dash) for an undirected edge."""
        net = abs(flow.edge_flow(u, v))
        is_bridge = u in flow.bottlenecks or v in flow.bottlenecks

        if net > 0:
            return "#4CAF50", 1.5 + (net / flow.max_flow) * 4, ""
        elif is_bridge:
            return "#E53935", 1.5, "4,3"
        else:
            return self.flag.darkPrimary, 1.0, "4,3"

    def edge_fragment(self, flow: SquadFlow,
                      u, v,
                      p1: MapCord, p2: MapCord,
                      builder,
                      shrink: float = 30.0) -> str:
        """SVG for one undirected edge: arrow + optional flow label."""
        dist = p1.distance(p2)
        if dist < 1: return ""

        net = flow.edge_flow(u, v)
        total = abs(net)
        src_p, dst_p = (p1, p2) if net >= 0 else (p2, p1)

        dx, dy = (dst_p.x - src_p.x) / dist, (dst_p.y - src_p.y) / dist
        start = MapCord(src_p.x + dx * shrink, src_p.y + dy * shrink)
        end   = MapCord(dst_p.x - dx * shrink, dst_p.y - dy * shrink)

        color, sw, dash = self.edge_color_width(flow, u, v)
        opacity = self.edge_opacity if total > 0 else self.edge_opacity * 0.4

        style = StyleCSS(f"sfr_edge_{u[:4]}_{v[:4]}",
                         stroke=color, stroke_width=sw,
                         fill=color, opacity=opacity,
                         **{"stroke-dasharray": dash} if dash else {})
        builder.add_style(style)

        parts = [MapPath([start, end], style).with_arrowhead(arrow_size=10, arrow_angle=25)]

        if total > 0:
            mid = start.midPoint(end)
            px, py = -dy * 12, dx * 12
            parts.append(
                f'<text x="{mid.x + px:.1f}" y="{mid.y + py:.1f}" '
                f'text-anchor="middle" font-size="10" font-weight="bold" '
                f'fill="#4CAF50" opacity="0.8">{total:.0f}🌾</text>')

        return '\n'.join(str(p) for p in parts)

    def all_edges(self, flow: SquadFlow,
                  svg_pos: dict[str, MapCord],
                  builder,
                  shrink: float = 30.0) -> str:
        """All edges, deduped."""
        seen = set()
        parts = []
        for u, v in flow.G.edges():
            key = tuple(sorted([u, v]))
            if key in seen: continue
            seen.add(key)
            parts.append(self.edge_fragment(flow, u, v,
                                            svg_pos[u], svg_pos[v],
                                            builder, shrink))
        return '\n'.join(parts)

    # ── Node helpers ─────────────────────────────────────────────────

    def node_ring_fragment(self, nid, data, flow: SquadFlow,
                           center: MapCord, builder) -> str:
        """Bottleneck outer ring + colored inner ring."""
        glyphs = DiagramGlyphs(self.flag, size=self.node_r)
        glyphs.register_styles(builder)
        ring_r = self.node_r + 6
        parts = []

        if nid in flow.bottlenecks:
            danger = glyphs.danger_style()
            builder.add_style(danger)
            parts.append(glyphs.ring(center, ring_r + 4, danger))

        post_net = flow.post_net(nid)
        ring_color = ("#FFD700" if data['is_king']
                      else "#4CAF50" if post_net >= 0
                      else "#E53935")
        ring_style = StyleCSS(f"sfr_ring_{nid[:6]}",
                              fill="white", stroke=ring_color,
                              stroke_width=3.5 if data['is_king'] else 2.5)
        builder.add_style(ring_style)
        parts.append(glyphs.ring(center, ring_r, ring_style))
        return '\n'.join(parts)

    def node_food_pip(self, nid, flow: SquadFlow,
                      center: MapCord, builder) -> str:
        """Food pip in upper-right of node."""
        glyphs = DiagramGlyphs(self.flag, size=self.node_r)
        glyphs.register_styles(builder)
        ring_r = self.node_r + 6
        post_net = flow.post_net(nid)
        frac = min(1.0, max(0.0, (post_net + 10) / 20))
        pip_center = MapCord(center.x + ring_r * 0.7, center.y - ring_r * 0.7)
        return glyphs.food_pip(pip_center, frac, max_r=8, min_r=4)

    def node_labels(self, piece, flow: SquadFlow, nid,
                    center: MapCord) -> str:
        """Name + stats text below node."""
        ring_r = self.node_r + 6
        post_net = flow.post_net(nid)
        pantry = piece.place.pantry if piece.place else 0
        tenure = piece.place.tenure if piece.place else 0
        life = piece.food.lifespan
        turns_left = max(0, life - tenure) if life < 999 else -1
        turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
        name_y = center.y + ring_r + 14
        net_color = "#4CAF50" if post_net >= 0 else "#E53935"

        parts = [
            f'<text x="{center.x:.1f}" y="{name_y:.1f}" '
            f'text-anchor="middle" font-size="14" fill="{self.flag.darkPrimary}">'
            f'{piece.name}</text>',
            f'<text x="{center.x:.1f}" y="{name_y+13:.1f}" '
            f'text-anchor="middle" font-size="12">'
            f'<tspan fill="{net_color}">{post_net:+.0f}🌾</tspan> '
            f'<tspan fill="#888">📦{pantry}</tspan> '
            f'<tspan fill="#666">{turns_str}</tspan></text>'
        ]
        return '\n'.join(parts)


@patch
def _render_edges(self: SquadFlowDiagram, flow, svg_pos, builder) -> str:
    r = SquadFlowRenderer(flag=self.squad.flag, node_r=22.5 * self.node_scale)
    return r.all_edges(flow, svg_pos, builder, shrink=22.5 * self.node_scale + 8)




@patch
def _render_node(self: SquadFlowDiagram, nid, data, flow, svg_pos, builder) -> str:
    r = SquadFlowRenderer(flag=self.squad.flag, node_r=22.5 * self.node_scale)
    center = svg_pos[nid]
    piece = data['piece']

    # Piece chess icon from flag
    svg_str, pat_def = self.squad.flag.piece_svg(
        piece.type, center,
        scale=self.node_scale, size='board',
        piece_id=f"flow_{nid}_{piece.type.value}")
    if pat_def is not None:
        builder.add_definition(pat_def)

    # Death marker if starving
    post_net = flow.post_net(nid)
    glyphs = DiagramGlyphs(self.squad.flag, size=22.5 * self.node_scale)
    glyphs.register_styles(builder)
    death = ""
    if post_net < 0 and not data['is_king']:
        ring_r = 22.5 * self.node_scale + 6
        death = glyphs.death_marker(
            MapCord(center.x - ring_r * 0.7, center.y - ring_r * 0.7),
            size=22.5 * self.node_scale * 0.4)

    hx_attrs = ""
    if self.on_click:
        hx_attrs = ' '.join(f'{k}="{v}"' for k, v in self.on_click(piece).items())

    parts = [f'<g style="cursor:pointer" {hx_attrs}>' if hx_attrs else '<g>',
             r.node_ring_fragment(nid, data, flow, center, builder),
             svg_str,
             death,
             r.node_food_pip(nid, flow, center, builder),
             r.node_labels(piece, flow, nid, center),
             '</g>']
    return '\n'.join(p for p in parts if p)


then what does the FoodNetworkOverlay look like?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The overlay slims down a lot — the main loop just resolves hex positions and delegates:

```python
def FoodNetworkOverlay(
    focus_squad: Squad | None = None,
    edge_opacity: float = 0.6,
    bottleneck_r_mult: float = 1.4,
    **kw
) -> OverlaySpec:

    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts = []

        for squad in ctx.squads:
            flow = SquadFlow.from_squad(squad)
            G = flow.G
            if len(G) < 1: continue

            flag = squad.flag
            is_focused = focus_squad is None or squad is focus_squad
            r = SquadFlowRenderer(flag=flag,
                                  node_r=grid.radius * 0.5,
                                  edge_opacity=edge_opacity)

            # ── Build svg_pos from hex grid centers ──────────────────
            svg_pos = {}
            for nid, data in G.nodes(data=True):
                idx = data['hex_idx']
                fis = [fi for fi in ctx.fine_indices(idx) if 0 <= fi < N]
                if fis:
                    svg_pos[nid] = grid.hexes[fis[0]].center

            missing = [n for n in G.nodes if n not in svg_pos]
            for n in missing:
                G.remove_node(n)  # off-grid pieces: skip silently

            if not svg_pos: continue

            # ── Edges ────────────────────────────────────────────────
            if is_focused and len(G) > 1:
                parts.append(r.all_edges(flow, svg_pos, ctx.builder,
                                         shrink=grid.radius * 0.4))

            # ── Target arrow (overlay-only, no equivalent in diagram) ─
            if is_focused:
                parts.append(_target_arrow(squad, G, svg_pos, grid, flag))

            # ── Nodes ────────────────────────────────────────────────
            for nid, data in G.nodes(data=True):
                if nid not in svg_pos: continue
                center = svg_pos[nid]
                piece  = data['piece']

                parts.append(r.node_ring_fragment(nid, data, flow, center, ctx.builder))
                parts.append(r.node_food_pip(nid, flow, center, ctx.builder))

                if is_focused:
                    parts.append(_pantry_bar(piece, flow, nid, center, grid))
                    parts.append(r.node_labels(piece, flow, nid, center))

        return '\n'.join(p for p in parts if p)

    return OverlaySpec("food_network", render, requires={'squads'}, priority=85)


def _target_arrow(squad, G, svg_pos, grid, flag) -> str:
    """Target direction indicator — overlay-specific, no diagram equivalent."""
    target = getattr(squad, '_target', None)
    if not target or target.hex_idx < 0: return ""
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id or king_id not in svg_pos: return ""

    kc = svg_pos[king_id]
    tc = grid.hexes[target.hex_idx].center
    dx, dy = tc.x - kc.x, tc.y - kc.y
    dist = math.sqrt(dx*dx + dy*dy)
    if dist < 1: return ""

    ux, uy = dx / dist, dy / dist
    arrow_len = grid.radius * 2.5
    ax, ay = kc.x + ux * arrow_len, kc.y + uy * arrow_len
    icon = "⚔️" if target.type == TargetType.KING else "🌾"
    sw = max(3, grid.radius * 0.15)
    fs = max(12, grid.radius * 0.5)

    return (f'<line x1="{kc.x:.1f}" y1="{kc.y:.1f}" x2="{ax:.1f}" y2="{ay:.1f}" '
            f'stroke="{flag.primary}" stroke-opacity="0.4" stroke-width="{sw:.1f}" '
            f'stroke-linecap="round" stroke-dasharray="8,4"/>'
            f'<text x="{ax:.1f}" y="{ay:.1f}" text-anchor="middle" '
            f'font-size="{fs:.0f}" dominant-baseline="central">{icon}</text>')


def _pantry_bar(piece, flow, nid, center: MapCord, grid) -> str:
    """Pantry fill bar — map overlay only (diagram uses footer instead)."""
    pantry   = piece.place.pantry if piece.place else 0
    post_net = flow.post_net(nid)
    bar_w    = grid.radius * 0.8
    bar_h    = max(3, grid.radius * 0.1)
    bar_x    = center.x - bar_w / 2
    bar_y    = center.y + grid.radius * 0.5 * 2.0
    max_p    = max(piece.food.cost * 2, 20)
    frac     = max(0.0, min(1.0, pantry / max_p))
    color    = "#4CAF50" if pantry > 0 else "#E53935"

    return (f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" width="{bar_w:.1f}" '
            f'height="{bar_h:.1f}" rx="1.5" fill="#ddd" opacity="0.6"/>'
            f'<rect x="{bar_x:.1f}" y="{bar_y:.1f}" width="{bar_w*frac:.1f}" '
            f'height="{bar_h:.1f}" rx="1.5" fill="{color}" opacity="0.8"/>')
```

The key differences from the diagram that stay in the overlay:
- **`svg_pos` from hex grid** instead of kamada-kawai
- **`_target_arrow`** — map-specific direction indicator
- **`_pantry_bar`** — map overlay uses this; diagram puts pantry in the footer text instead
- **`fine_indices` resolution** — coarse→fine hex mapping, map-only concept

node_labels should use badges and be cleaned up. it is very cluttered are things like progress bars or vertical ones better?
```python
                   center: MapCord) -> str:
        """Name + stats text below node."""
        ring_r = self.node_r + 6
        post_net = flow.post_net(nid)
        pantry = piece.place.pantry if piece.place else 0
        tenure = piece.place.tenure if piece.place else 0
        life = piece.food.lifespan
        turns_left = max(0, life - tenure) if life < 999 else -1
        turns_str = "∞" if turns_left < 0 else f"{turns_left}t"
        name_y = center.y + ring_r + 14
        net_color = "#4CAF50" if post_net >= 0 else "#E53935"

        parts = [
            f'<text x="{center.x:.1f}" y="{name_y:.1f}" '
            f'text-anchor="middle" font-size="14" fill="{self.flag.darkPrimary}">'
            f'{piece.name}</text>',
            f'<text x="{center.x:.1f}" y="{name_y+16:.1f}" '
            f'text-anchor="middle" font-size="12">'
            f'<tspan fill="{net_color}" font-size="15" font-weight="700">{post_net:+.0f}🌾</tspan> '
            f'<tspan fill="{self.flag.darkPrimary}" font-size="15" font-weight="700">🧺{pantry}</tspan> '
            f'<tspan fill="#666">{turns_str}</tspan></text>'
        ]

        return '\n'.join(parts)
    ```
    also maybe more spacing

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Since this is SVG (not HTML), we can't use `FT` badges directly, but we can make SVG **pill badges** — `<rect>` + `<text>` combos — and stack them vertically with proper spacing. Much cleaner than cramming tspans on one line:

```python
def node_labels(self, piece, flow: SquadFlow, nid, center: MapCord) -> str:
    """Name + vertically stacked SVG pill badges below node."""
    ring_r   = self.node_r + 6
    post_net = flow.post_net(nid)
    pantry   = piece.place.pantry if piece.place else 0
    tenure   = piece.place.tenure if piece.place else 0
    life     = piece.food.lifespan
    turns_left = max(0, life - tenure) if life < 999 else -1
    turns_str  = "∞" if turns_left < 0 else f"{turns_left}t"

    net_color  = "#4CAF50" if post_net >= 0 else "#E53935"
    net_bg     = "#e8f5e9" if post_net >= 0 else "#ffebee"
    pant_color = self.flag.darkPrimary

    cx = center.x
    y  = center.y + ring_r + 18  # start below ring
    dy = 22                       # vertical spacing between badges

    def pill(label, fg, bg, y_pos, w=70, h=16, r=8):
        return (
            f'<rect x="{cx - w/2:.1f}" y="{y_pos - h/2 - 1:.1f}" '
            f'width="{w}" height="{h}" rx="{r}" fill="{bg}" opacity="0.85"/>'
            f'<text x="{cx:.1f}" y="{y_pos:.1f}" text-anchor="middle" '
            f'dominant-baseline="central" font-size="12" font-weight="700" fill="{fg}">'
            f'{label}</text>'
        )

    parts = [
        # Name — no pill, just bold text
        f'<text x="{cx:.1f}" y="{y:.1f}" text-anchor="middle" '
        f'dominant-baseline="central" font-size="13" font-weight="700" '
        f'fill="{self.flag.darkPrimary}">{piece.name}</text>',

        # Net food pill
        pill(f"{post_net:+.0f} 🌾", net_color, net_bg, y + dy),

        # Pantry pill
        pill(f"🧺 {pantry}", pant_color, "#f5f5f5", y + dy * 2),

        # Turns pill — only show if finite
        *([] if turns_left < 0 else
          [pill(f"⏳ {turns_str}", "#666", "#f5f5f5", y + dy * 3, w=55)]),
    ]

    return '\n'.join(parts)
```

Key improvements:
- **Vertical stack** with `dy=22` spacing — no more cramped single line
- **Pill backgrounds** (`<rect rx>` + `<text>`) make each stat visually distinct
- **Name stays plain** — bold, no pill, acts as header
- **Turns hidden when infinite** — kings don't need a ⏳ badge cluttering things
- `dominant-baseline="central"` centres text in pill vertically without manual offset fiddling

In [ ]:
show(squad_flow_demo())

In [ ]:
from fasthtml.common import *
from fasthtml.jupyter import *
from fasthtml.common import *